# Multi-Temporal LULC Classification & Explainability Pipeline (2020, 2022, 2024)
================================================================================
**Study Area**: Greater Accra Metropolitan Area (GAMA), Ghana  
**Algorithms**: Random Forest (RF), SVM, XGBoost  
**Explainability**: SHAP (SHapley Additive exPlanations) on the winning algorithm  
================================================================================

In [ ]:
!pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn rasterio --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
import warnings
import os
import glob

# Try importing rasterio (essential for GIS/RS spatial representation)
try:
    import rasterio
    from rasterio.plot import show
    RASTERIO_AVAILABLE = True
except ImportError:
    RASTERIO_AVAILABLE = False
    print("Warning: 'rasterio' not found. Spatial map generation will be skipped. Run: !pip install rasterio")

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    cohen_kappa_score,
    f1_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# SHAP initialization for notebook plots
shap.initjs()

print("All dependencies loaded successfully.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Global Setup ---
DATA_DIR = '/content/drive/MyDrive/XAI_LULC_Ghana/'
if not os.path.exists(DATA_DIR):
    DATA_DIR = r'c:\YERBO\Desktop\CODING\EXPLAINABLE AI\\'

CLASS_NAMES = {
    1: 'Built-Up',
    2: 'Vegetation',
    3: 'Water',
    4: 'Bare Soil'
}

CLASS_COLORS = {
    1: '#E8453C',   # Built-Up (red)
    2: '#1B7A2F',   # Vegetation (dark green)
    3: '#2196F3',   # Water (blue)
    4: '#9E9E9E'    # Bare Soil (grey)
}

# Redundant features to drop based on multicollinearity control
COLS_TO_DROP = [
    'B2',         # Drop Blue (r = 0.9560 with Green B3) -> Keep B3
    'B5',         # Drop RedEdge-1 (r = 0.9539 with Red B4) -> Keep B4
    'B6',         # Drop RedEdge-2 (r = 0.9787 with NIR B8) -> Keep B8
    'B7',         # Drop RedEdge-3 (r = 0.9929 with NIR B8) -> Keep B8
    'B8A',        # Drop Narrow NIR (r = 0.9936 with NIR B8) -> Keep B8
    'B12',        # Drop SWIR-2 (r = 0.9646 with SWIR-1 B11) -> Keep B11
    'EVI',        # Drop EVI (r = 0.9670 with NDVI) -> Keep NDVI
    'SAVI',       # Drop SAVI (r = 0.9735 with NDVI) -> Keep NDVI
    'NDWI',       # Drop NDWI (r = -0.9520 with NDVI) -> Keep MNDWI (better for urban areas)
    'GLCM_ASM'    # Drop GLCM_ASM (r = -0.9708 with GLCM_Entropy) -> Keep GLCM_Entropy
]

## Step 3: Run Year 2020 Pipeline

In [ ]:
# ============================================================================
# YEAR 2020 DATA PREPROCESSING & MODEL TRAINING
# ============================================================================

DATA_PATH_2020 = os.path.join(DATA_DIR, 'XAI_TRAINING_DATASET_2020.csv')
if not os.path.exists(DATA_PATH_2020) and False:
    fallback_path = os.path.join(DATA_DIR, 'XAI TRAINING DATASET.csv')
    if os.path.exists(fallback_path):
        DATA_PATH_2020 = fallback_path

if os.path.exists(DATA_PATH_2020):
    year_2020_available = True
    print(f"\nProcessing dataset for 2020 LULC...")
    df_2020 = pd.read_csv(DATA_PATH_2020)
    print(f"Dataset shape: {df_2020.shape}")
    
    # Identify target class column
    LABEL_COL_2020 = None
    for col in ['class', 'LULC_CLASS']:
        if col in df_2020.columns:
            LABEL_COL_2020 = col
            break
    if LABEL_COL_2020 is None:
        LABEL_COL_2020 = df_2020.columns[-1]
        print(f"Warning: Target column not found. Using '{LABEL_COL_2020}'.")
        
    # Drop GEE system metadata columns
    for col in ['system:index', '.geo']:
        if col in df_2020.columns:
            df_2020 = df_2020.drop(columns=[col])
            print(f"Dropped metadata column: {col}")
            
    # Remap 1-4 LULC classes to 0-3 for XGBoost compatibility
    df_2020['label_0idx'] = df_2020[LABEL_COL_2020] - 1
    NUM_CLASSES_2020 = df_2020['label_0idx'].nunique()
    
    # Feature columns
    FEATURE_COLS_2020 = [col for col in df_2020.columns if col not in [LABEL_COL_2020, 'label_0idx']]
    
    # Class distribution print
    print(f"\n--- Class Distribution (2020) ---")
    class_counts_2020 = df_2020[LABEL_COL_2020].value_counts().sort_index()
    for cls_id, count in class_counts_2020.items():
        print(f"  Class {cls_id} ({CLASS_NAMES[cls_id]}): {count} samples")
    print(f"  TOTAL: {class_counts_2020.sum()} samples")
    
    # Plot Class Distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        [CLASS_NAMES[i] for i in class_counts_2020.index],
        class_counts_2020.values,
        color=[CLASS_COLORS[i] for i in class_counts_2020.index],
        edgecolor='black', linewidth=0.5
    )
    ax.set_title('Training Sample Distribution by LULC Class (2020)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Samples')
    for bar, count in zip(bars, class_counts_2020.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig('class_distribution_2020.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Missing values check
    print(f"\nMissing values: {df_2020[FEATURE_COLS_2020].isnull().sum().sum()}")
    df_2020 = df_2020.dropna(subset=FEATURE_COLS_2020)
    
    # Heatmap Feature Correlation
    corr_matrix_2020 = df_2020[FEATURE_COLS_2020].corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix_2020, dtype=bool))
    sns.heatmap(corr_matrix_2020, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7})
    ax.set_title('Feature Correlation Matrix (2020)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_correlation_2020.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Flag correlated pairs
    high_corr_2020 = []
    for i in range(len(corr_matrix_2020.columns)):
        for j in range(i+1, len(corr_matrix_2020.columns)):
            if abs(corr_matrix_2020.iloc[i, j]) > 0.95:
                high_corr_2020.append((corr_matrix_2020.columns[i], corr_matrix_2020.columns[j], corr_matrix_2020.iloc[i, j]))
    if high_corr_2020:
        print("\n⚠ Highly correlated feature pairs (|r| > 0.95):")
        for f1, f2, r in high_corr_2020:
            print(f"  {f1} ↔ {f2}: r = {r:.4f}")
            
    # Drop features
    FEATURE_COLS_2020 = [col for col in FEATURE_COLS_2020 if col not in COLS_TO_DROP]
    
    X_2020 = df_2020[FEATURE_COLS_2020].values
    y_2020 = df_2020['label_0idx'].values
    
    # Split
    X_train_2020, X_temp_2020, y_train_2020, y_temp_2020 = train_test_split(
        X_2020, y_2020, test_size=0.30, random_state=42, stratify=y_2020
    )
    X_val_2020, X_test_2020, y_val_2020, y_test_2020 = train_test_split(
        X_temp_2020, y_temp_2020, test_size=0.50, random_state=42, stratify=y_temp_2020
    )
    
    print(f"\nTraining set:   {X_train_2020.shape[0]} samples")
    print(f"Validation set: {X_val_2020.shape[0]} samples")
    print(f"Test set:       {X_test_2020.shape[0]} samples")
    
    # Verify class balance
    for name, ys in [('Train', y_train_2020), ('Val', y_val_2020), ('Test', y_test_2020)]:
        unique, counts = np.unique(ys, return_counts=True)
        dist = ', '.join([f"{CLASS_NAMES[u+1]}: {c}" for u, c in zip(unique, counts)])
        print(f"  {name}: {dist}")
        
    # --- MODEL TRAINING ---
    print("\nTraining Classifiers (Random Forest, SVM, XGBoost) for 2020...")
    
    # 1. Random Forest
    rf_2020 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_2020.fit(X_train_2020, y_train_2020)
    y_pred_rf_2020 = rf_2020.predict(X_test_2020)
    oa_rf_2020 = accuracy_score(y_test_2020, y_pred_rf_2020)
    kappa_rf_2020 = cohen_kappa_score(y_test_2020, y_pred_rf_2020)
    f1_rf_2020 = f1_score(y_test_2020, y_pred_rf_2020, average='weighted')
    
    # 2. SVM
    svm_2020 = SVC(probability=True, kernel='rbf', C=1.0, random_state=42)
    svm_2020.fit(X_train_2020, y_train_2020)
    y_pred_svm_2020 = svm_2020.predict(X_test_2020)
    oa_svm_2020 = accuracy_score(y_test_2020, y_pred_svm_2020)
    kappa_svm_2020 = cohen_kappa_score(y_test_2020, y_pred_svm_2020)
    f1_svm_2020 = f1_score(y_test_2020, y_pred_svm_2020, average='weighted')
    
    # 3. XGBoost
    xgb_params_2020 = {
        'objective': 'multi:softprob',
        'num_class': NUM_CLASSES_2020,
        'eval_metric': 'mlogloss',
        'max_depth': 8,
        'learning_rate': 0.1,
        'n_estimators': 300,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 1
    }
    xgb_2020 = xgb.XGBClassifier(**xgb_params_2020)
    xgb_2020.fit(
        X_train_2020, y_train_2020,
        eval_set=[(X_train_2020, y_train_2020), (X_val_2020, y_val_2020)],
        verbose=False
    )
    y_pred_xgb_2020 = xgb_2020.predict(X_test_2020)
    oa_xgb_2020 = accuracy_score(y_test_2020, y_pred_xgb_2020)
    kappa_xgb_2020 = cohen_kappa_score(y_test_2020, y_pred_xgb_2020)
    f1_xgb_2020 = f1_score(y_test_2020, y_pred_xgb_2020, average='weighted')
    
    print(f"\n--- Model Accuracy Metrics for 2020 ---")
    print(f"  Random Forest -> OA: {oa_rf_2020:.4f} | Kappa: {kappa_rf_2020:.4f} | Weighted F1: {f1_rf_2020:.4f}")
    print(f"  SVM           -> OA: {oa_svm_2020:.4f} | Kappa: {kappa_svm_2020:.4f} | Weighted F1: {f1_svm_2020:.4f}")
    print(f"  XGBoost       -> OA: {oa_xgb_2020:.4f} | Kappa: {kappa_xgb_2020:.4f} | Weighted F1: {f1_xgb_2020:.4f}")
    
else:
    year_2020_available = False
    print("Warning: Dataset for 2020 not found. Skipping 2020 preprocessing & model training.")



## Step 4: Run Year 2022 Pipeline

In [ ]:
# ============================================================================
# YEAR 2022 DATA PREPROCESSING & MODEL TRAINING
# ============================================================================

DATA_PATH_2022 = os.path.join(DATA_DIR, 'XAI_TRAINING_DATASET_2022.csv')
if not os.path.exists(DATA_PATH_2022) and False:
    fallback_path = os.path.join(DATA_DIR, 'XAI TRAINING DATASET.csv')
    if os.path.exists(fallback_path):
        DATA_PATH_2022 = fallback_path

if os.path.exists(DATA_PATH_2022):
    year_2022_available = True
    print(f"\nProcessing dataset for 2022 LULC...")
    df_2022 = pd.read_csv(DATA_PATH_2022)
    print(f"Dataset shape: {df_2022.shape}")
    
    # Identify target class column
    LABEL_COL_2022 = None
    for col in ['class', 'LULC_CLASS']:
        if col in df_2022.columns:
            LABEL_COL_2022 = col
            break
    if LABEL_COL_2022 is None:
        LABEL_COL_2022 = df_2022.columns[-1]
        print(f"Warning: Target column not found. Using '{LABEL_COL_2022}'.")
        
    # Drop GEE system metadata columns
    for col in ['system:index', '.geo']:
        if col in df_2022.columns:
            df_2022 = df_2022.drop(columns=[col])
            print(f"Dropped metadata column: {col}")
            
    # Remap 1-4 LULC classes to 0-3 for XGBoost compatibility
    df_2022['label_0idx'] = df_2022[LABEL_COL_2022] - 1
    NUM_CLASSES_2022 = df_2022['label_0idx'].nunique()
    
    # Feature columns
    FEATURE_COLS_2022 = [col for col in df_2022.columns if col not in [LABEL_COL_2022, 'label_0idx']]
    
    # Class distribution print
    print(f"\n--- Class Distribution (2022) ---")
    class_counts_2022 = df_2022[LABEL_COL_2022].value_counts().sort_index()
    for cls_id, count in class_counts_2022.items():
        print(f"  Class {cls_id} ({CLASS_NAMES[cls_id]}): {count} samples")
    print(f"  TOTAL: {class_counts_2022.sum()} samples")
    
    # Plot Class Distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        [CLASS_NAMES[i] for i in class_counts_2022.index],
        class_counts_2022.values,
        color=[CLASS_COLORS[i] for i in class_counts_2022.index],
        edgecolor='black', linewidth=0.5
    )
    ax.set_title('Training Sample Distribution by LULC Class (2022)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Samples')
    for bar, count in zip(bars, class_counts_2022.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig('class_distribution_2022.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Missing values check
    print(f"\nMissing values: {df_2022[FEATURE_COLS_2022].isnull().sum().sum()}")
    df_2022 = df_2022.dropna(subset=FEATURE_COLS_2022)
    
    # Heatmap Feature Correlation
    corr_matrix_2022 = df_2022[FEATURE_COLS_2022].corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix_2022, dtype=bool))
    sns.heatmap(corr_matrix_2022, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7})
    ax.set_title('Feature Correlation Matrix (2022)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_correlation_2022.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Flag correlated pairs
    high_corr_2022 = []
    for i in range(len(corr_matrix_2022.columns)):
        for j in range(i+1, len(corr_matrix_2022.columns)):
            if abs(corr_matrix_2022.iloc[i, j]) > 0.95:
                high_corr_2022.append((corr_matrix_2022.columns[i], corr_matrix_2022.columns[j], corr_matrix_2022.iloc[i, j]))
    if high_corr_2022:
        print("\n⚠ Highly correlated feature pairs (|r| > 0.95):")
        for f1, f2, r in high_corr_2022:
            print(f"  {f1} ↔ {f2}: r = {r:.4f}")
            
    # Drop features
    FEATURE_COLS_2022 = [col for col in FEATURE_COLS_2022 if col not in COLS_TO_DROP]
    
    X_2022 = df_2022[FEATURE_COLS_2022].values
    y_2022 = df_2022['label_0idx'].values
    
    # Split
    X_train_2022, X_temp_2022, y_train_2022, y_temp_2022 = train_test_split(
        X_2022, y_2022, test_size=0.30, random_state=42, stratify=y_2022
    )
    X_val_2022, X_test_2022, y_val_2022, y_test_2022 = train_test_split(
        X_temp_2022, y_temp_2022, test_size=0.50, random_state=42, stratify=y_temp_2022
    )
    
    print(f"\nTraining set:   {X_train_2022.shape[0]} samples")
    print(f"Validation set: {X_val_2022.shape[0]} samples")
    print(f"Test set:       {X_test_2022.shape[0]} samples")
    
    # Verify class balance
    for name, ys in [('Train', y_train_2022), ('Val', y_val_2022), ('Test', y_test_2022)]:
        unique, counts = np.unique(ys, return_counts=True)
        dist = ', '.join([f"{CLASS_NAMES[u+1]}: {c}" for u, c in zip(unique, counts)])
        print(f"  {name}: {dist}")
        
    # --- MODEL TRAINING ---
    print("\nTraining Classifiers (Random Forest, SVM, XGBoost) for 2022...")
    
    # 1. Random Forest
    rf_2022 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_2022.fit(X_train_2022, y_train_2022)
    y_pred_rf_2022 = rf_2022.predict(X_test_2022)
    oa_rf_2022 = accuracy_score(y_test_2022, y_pred_rf_2022)
    kappa_rf_2022 = cohen_kappa_score(y_test_2022, y_pred_rf_2022)
    f1_rf_2022 = f1_score(y_test_2022, y_pred_rf_2022, average='weighted')
    
    # 2. SVM
    svm_2022 = SVC(probability=True, kernel='rbf', C=1.0, random_state=42)
    svm_2022.fit(X_train_2022, y_train_2022)
    y_pred_svm_2022 = svm_2022.predict(X_test_2022)
    oa_svm_2022 = accuracy_score(y_test_2022, y_pred_svm_2022)
    kappa_svm_2022 = cohen_kappa_score(y_test_2022, y_pred_svm_2022)
    f1_svm_2022 = f1_score(y_test_2022, y_pred_svm_2022, average='weighted')
    
    # 3. XGBoost
    xgb_params_2022 = {
        'objective': 'multi:softprob',
        'num_class': NUM_CLASSES_2022,
        'eval_metric': 'mlogloss',
        'max_depth': 8,
        'learning_rate': 0.1,
        'n_estimators': 300,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 1
    }
    xgb_2022 = xgb.XGBClassifier(**xgb_params_2022)
    xgb_2022.fit(
        X_train_2022, y_train_2022,
        eval_set=[(X_train_2022, y_train_2022), (X_val_2022, y_val_2022)],
        verbose=False
    )
    y_pred_xgb_2022 = xgb_2022.predict(X_test_2022)
    oa_xgb_2022 = accuracy_score(y_test_2022, y_pred_xgb_2022)
    kappa_xgb_2022 = cohen_kappa_score(y_test_2022, y_pred_xgb_2022)
    f1_xgb_2022 = f1_score(y_test_2022, y_pred_xgb_2022, average='weighted')
    
    print(f"\n--- Model Accuracy Metrics for 2022 ---")
    print(f"  Random Forest -> OA: {oa_rf_2022:.4f} | Kappa: {kappa_rf_2022:.4f} | Weighted F1: {f1_rf_2022:.4f}")
    print(f"  SVM           -> OA: {oa_svm_2022:.4f} | Kappa: {kappa_svm_2022:.4f} | Weighted F1: {f1_svm_2022:.4f}")
    print(f"  XGBoost       -> OA: {oa_xgb_2022:.4f} | Kappa: {kappa_xgb_2022:.4f} | Weighted F1: {f1_xgb_2022:.4f}")
    
else:
    year_2022_available = False
    print("Warning: Dataset for 2022 not found. Skipping 2022 preprocessing & model training.")



## Step 5: Run Year 2024 Pipeline

In [ ]:
# ============================================================================
# YEAR 2024 DATA PREPROCESSING & MODEL TRAINING
# ============================================================================

DATA_PATH_2024 = os.path.join(DATA_DIR, 'XAI_TRAINING_DATASET_2024.csv')
if not os.path.exists(DATA_PATH_2024) and True:
    fallback_path = os.path.join(DATA_DIR, 'XAI TRAINING DATASET.csv')
    if os.path.exists(fallback_path):
        DATA_PATH_2024 = fallback_path

if os.path.exists(DATA_PATH_2024):
    year_2024_available = True
    print(f"\nProcessing dataset for 2024 LULC...")
    df_2024 = pd.read_csv(DATA_PATH_2024)
    print(f"Dataset shape: {df_2024.shape}")
    
    # Identify target class column
    LABEL_COL_2024 = None
    for col in ['class', 'LULC_CLASS']:
        if col in df_2024.columns:
            LABEL_COL_2024 = col
            break
    if LABEL_COL_2024 is None:
        LABEL_COL_2024 = df_2024.columns[-1]
        print(f"Warning: Target column not found. Using '{LABEL_COL_2024}'.")
        
    # Drop GEE system metadata columns
    for col in ['system:index', '.geo']:
        if col in df_2024.columns:
            df_2024 = df_2024.drop(columns=[col])
            print(f"Dropped metadata column: {col}")
            
    # Remap 1-4 LULC classes to 0-3 for XGBoost compatibility
    df_2024['label_0idx'] = df_2024[LABEL_COL_2024] - 1
    NUM_CLASSES_2024 = df_2024['label_0idx'].nunique()
    
    # Feature columns
    FEATURE_COLS_2024 = [col for col in df_2024.columns if col not in [LABEL_COL_2024, 'label_0idx']]
    
    # Class distribution print
    print(f"\n--- Class Distribution (2024) ---")
    class_counts_2024 = df_2024[LABEL_COL_2024].value_counts().sort_index()
    for cls_id, count in class_counts_2024.items():
        print(f"  Class {cls_id} ({CLASS_NAMES[cls_id]}): {count} samples")
    print(f"  TOTAL: {class_counts_2024.sum()} samples")
    
    # Plot Class Distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        [CLASS_NAMES[i] for i in class_counts_2024.index],
        class_counts_2024.values,
        color=[CLASS_COLORS[i] for i in class_counts_2024.index],
        edgecolor='black', linewidth=0.5
    )
    ax.set_title('Training Sample Distribution by LULC Class (2024)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Samples')
    for bar, count in zip(bars, class_counts_2024.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig('class_distribution_2024.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Missing values check
    print(f"\nMissing values: {df_2024[FEATURE_COLS_2024].isnull().sum().sum()}")
    df_2024 = df_2024.dropna(subset=FEATURE_COLS_2024)
    
    # Heatmap Feature Correlation
    corr_matrix_2024 = df_2024[FEATURE_COLS_2024].corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix_2024, dtype=bool))
    sns.heatmap(corr_matrix_2024, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7})
    ax.set_title('Feature Correlation Matrix (2024)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_correlation_2024.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Flag correlated pairs
    high_corr_2024 = []
    for i in range(len(corr_matrix_2024.columns)):
        for j in range(i+1, len(corr_matrix_2024.columns)):
            if abs(corr_matrix_2024.iloc[i, j]) > 0.95:
                high_corr_2024.append((corr_matrix_2024.columns[i], corr_matrix_2024.columns[j], corr_matrix_2024.iloc[i, j]))
    if high_corr_2024:
        print("\n⚠ Highly correlated feature pairs (|r| > 0.95):")
        for f1, f2, r in high_corr_2024:
            print(f"  {f1} ↔ {f2}: r = {r:.4f}")
            
    # Drop features
    FEATURE_COLS_2024 = [col for col in FEATURE_COLS_2024 if col not in COLS_TO_DROP]
    
    X_2024 = df_2024[FEATURE_COLS_2024].values
    y_2024 = df_2024['label_0idx'].values
    
    # Split
    X_train_2024, X_temp_2024, y_train_2024, y_temp_2024 = train_test_split(
        X_2024, y_2024, test_size=0.30, random_state=42, stratify=y_2024
    )
    X_val_2024, X_test_2024, y_val_2024, y_test_2024 = train_test_split(
        X_temp_2024, y_temp_2024, test_size=0.50, random_state=42, stratify=y_temp_2024
    )
    
    print(f"\nTraining set:   {X_train_2024.shape[0]} samples")
    print(f"Validation set: {X_val_2024.shape[0]} samples")
    print(f"Test set:       {X_test_2024.shape[0]} samples")
    
    # Verify class balance
    for name, ys in [('Train', y_train_2024), ('Val', y_val_2024), ('Test', y_test_2024)]:
        unique, counts = np.unique(ys, return_counts=True)
        dist = ', '.join([f"{CLASS_NAMES[u+1]}: {c}" for u, c in zip(unique, counts)])
        print(f"  {name}: {dist}")
        
    # --- MODEL TRAINING ---
    print("\nTraining Classifiers (Random Forest, SVM, XGBoost) for 2024...")
    
    # 1. Random Forest
    rf_2024 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_2024.fit(X_train_2024, y_train_2024)
    y_pred_rf_2024 = rf_2024.predict(X_test_2024)
    oa_rf_2024 = accuracy_score(y_test_2024, y_pred_rf_2024)
    kappa_rf_2024 = cohen_kappa_score(y_test_2024, y_pred_rf_2024)
    f1_rf_2024 = f1_score(y_test_2024, y_pred_rf_2024, average='weighted')
    
    # 2. SVM
    svm_2024 = SVC(probability=True, kernel='rbf', C=1.0, random_state=42)
    svm_2024.fit(X_train_2024, y_train_2024)
    y_pred_svm_2024 = svm_2024.predict(X_test_2024)
    oa_svm_2024 = accuracy_score(y_test_2024, y_pred_svm_2024)
    kappa_svm_2024 = cohen_kappa_score(y_test_2024, y_pred_svm_2024)
    f1_svm_2024 = f1_score(y_test_2024, y_pred_svm_2024, average='weighted')
    
    # 3. XGBoost
    xgb_params_2024 = {
        'objective': 'multi:softprob',
        'num_class': NUM_CLASSES_2024,
        'eval_metric': 'mlogloss',
        'max_depth': 8,
        'learning_rate': 0.1,
        'n_estimators': 300,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 1
    }
    xgb_2024 = xgb.XGBClassifier(**xgb_params_2024)
    xgb_2024.fit(
        X_train_2024, y_train_2024,
        eval_set=[(X_train_2024, y_train_2024), (X_val_2024, y_val_2024)],
        verbose=False
    )
    y_pred_xgb_2024 = xgb_2024.predict(X_test_2024)
    oa_xgb_2024 = accuracy_score(y_test_2024, y_pred_xgb_2024)
    kappa_xgb_2024 = cohen_kappa_score(y_test_2024, y_pred_xgb_2024)
    f1_xgb_2024 = f1_score(y_test_2024, y_pred_xgb_2024, average='weighted')
    
    print(f"\n--- Model Accuracy Metrics for 2024 ---")
    print(f"  Random Forest -> OA: {oa_rf_2024:.4f} | Kappa: {kappa_rf_2024:.4f} | Weighted F1: {f1_rf_2024:.4f}")
    print(f"  SVM           -> OA: {oa_svm_2024:.4f} | Kappa: {kappa_svm_2024:.4f} | Weighted F1: {f1_svm_2024:.4f}")
    print(f"  XGBoost       -> OA: {oa_xgb_2024:.4f} | Kappa: {kappa_xgb_2024:.4f} | Weighted F1: {f1_xgb_2024:.4f}")
    
else:
    year_2024_available = False
    print("Warning: Dataset for 2024 not found. Skipping 2024 preprocessing & model training.")



## Step 6: Compare Performance Across Years and Select the Winner Algorithm

In [ ]:
comparison_records = []
oa_rf_list = []
oa_svm_list = []
oa_xgb_list = []

if year_2020_available:
    comparison_records.append({'Year': 2020, 'Algorithm': 'Random Forest', 'OA': oa_rf_2020, 'Kappa': kappa_rf_2020, 'Weighted F1': f1_rf_2020})
    comparison_records.append({'Year': 2020, 'Algorithm': 'SVM', 'OA': oa_svm_2020, 'Kappa': kappa_svm_2020, 'Weighted F1': f1_svm_2020})
    comparison_records.append({'Year': 2020, 'Algorithm': 'XGBoost', 'OA': oa_xgb_2020, 'Kappa': kappa_xgb_2020, 'Weighted F1': f1_xgb_2020})
    oa_rf_list.append(oa_rf_2020)
    oa_svm_list.append(oa_svm_2020)
    oa_xgb_list.append(oa_xgb_2020)

if year_2022_available:
    comparison_records.append({'Year': 2022, 'Algorithm': 'Random Forest', 'OA': oa_rf_2022, 'Kappa': kappa_rf_2022, 'Weighted F1': f1_rf_2022})
    comparison_records.append({'Year': 2022, 'Algorithm': 'SVM', 'OA': oa_svm_2022, 'Kappa': kappa_svm_2022, 'Weighted F1': f1_svm_2022})
    comparison_records.append({'Year': 2022, 'Algorithm': 'XGBoost', 'OA': oa_xgb_2022, 'Kappa': kappa_xgb_2022, 'Weighted F1': f1_xgb_2022})
    oa_rf_list.append(oa_rf_2022)
    oa_svm_list.append(oa_svm_2022)
    oa_xgb_list.append(oa_xgb_2022)

if year_2024_available:
    comparison_records.append({'Year': 2024, 'Algorithm': 'Random Forest', 'OA': oa_rf_2024, 'Kappa': kappa_rf_2024, 'Weighted F1': f1_rf_2024})
    comparison_records.append({'Year': 2024, 'Algorithm': 'SVM', 'OA': oa_svm_2024, 'Kappa': kappa_svm_2024, 'Weighted F1': f1_svm_2024})
    comparison_records.append({'Year': 2024, 'Algorithm': 'XGBoost', 'OA': oa_xgb_2024, 'Kappa': kappa_xgb_2024, 'Weighted F1': f1_xgb_2024})
    oa_rf_list.append(oa_rf_2024)
    oa_svm_list.append(oa_svm_2024)
    oa_xgb_list.append(oa_xgb_2024)

if comparison_records:
    comp_df = pd.DataFrame(comparison_records)
    print("\n" + "="*80)
    print(" CROSS-YEAR ALGORITHM PERFORMANCE SUMMARY")
    print("="*80)
    print(comp_df.to_string(index=False))
    comp_df.to_csv(os.path.join(DATA_DIR, 'algorithm_comparison_report.csv'), index=False)
    
    # Find winner algorithm based on highest mean OA
    rf_mean = np.mean(oa_rf_list) if oa_rf_list else 0.0
    svm_mean = np.mean(oa_svm_list) if oa_svm_list else 0.0
    xgb_mean = np.mean(oa_xgb_list) if oa_xgb_list else 0.0
    
    print(f"\nMean OA - Random Forest: {rf_mean:.4f}")
    print(f"Mean OA - SVM:           {svm_mean:.4f}")
    print(f"Mean OA - XGBoost:       {xgb_mean:.4f}")
    
    if rf_mean >= svm_mean and rf_mean >= xgb_mean:
        best_algorithm = 'Random Forest'
    elif svm_mean >= rf_mean and svm_mean >= xgb_mean:
        best_algorithm = 'SVM'
    else:
        best_algorithm = 'XGBoost'
        
    print(f"\n🏆 Overall Winner Algorithm: {best_algorithm} (Average OA: {max(rf_mean, svm_mean, xgb_mean):.4f})")
else:
    print("No datasets were processed. Cannot select winner algorithm.")

## Step 7: Explainable AI (SHAP) - Year 2020

In [ ]:
# ============================================================================
# YEAR 2020 SHAP EXPLAINABILITY (Using Winning Algorithm: {best_algorithm})
# ============================================================================

if year_2020_available:
    print(f"\nComputing SHAP values for 2020 using {best_algorithm}...")
    
    # Retrieve winning model and test set for 2020
    if best_algorithm == 'Random Forest':
        model_2020 = rf_2020
    elif best_algorithm == 'SVM':
        model_2020 = svm_2020
    else:
        model_2020 = xgb_2020
        
    X_test_df_2020 = pd.DataFrame(X_test_2020, columns=FEATURE_COLS_2020)
    
    # TreeExplainer is fast for tree-based models; KernelExplainer is used for SVM
    if best_algorithm in ['XGBoost', 'Random Forest']:
        explainer_2020 = shap.TreeExplainer(model_2020)
        shap_values_2020 = explainer_2020.shap_values(X_test_2020)
    else:
        X_train_summary_2020 = shap.kmeans(X_train_2020, 20)
        explainer_2020 = shap.KernelExplainer(model_2020.predict_proba, X_train_summary_2020)
        shap_values_2020 = explainer_2020.shap_values(X_test_2020)
        
    print(f"✓ SHAP values computed for 2020 test samples.")
    if isinstance(shap_values_2020, list):
        print(f"  Number of class arrays: {len(shap_values_2020)}")
        print(f"  Shape per class: {shap_values_2020[0].shape}")
    elif isinstance(shap_values_2020, np.ndarray):
        print(f"  Shape: {shap_values_2020.shape}")
        
    # --- 9A. SHAP Bar Plot — Mean |SHAP| per feature across ALL classes ---
    print(f"\n--- SHAP Global Feature Importance (2020) ---")
    fig = plt.figure(figsize=(12, 8))
    target_names_2020 = [CLASS_NAMES[i+1] for i in range(NUM_CLASSES_2020)]
    shap.summary_plot(shap_values_2020, X_test_df_2020, plot_type='bar',
                      class_names=target_names_2020, show=False)
    plt.title(f'SHAP Feature Importance — All LULC Classes (2020 - {best_algorithm})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_summary_bar_all_classes_2020.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9B. SHAP Beeswarm Plot — Per Class ---
    for cls_idx in range(NUM_CLASSES_2020):
        cls_name = CLASS_NAMES[cls_idx + 1]
        print(f"\n--- SHAP Beeswarm: {cls_name} (2020) ---")
        fig = plt.figure(figsize=(12, 8))
        if isinstance(shap_values_2020, list):
            sv = shap_values_2020[cls_idx]
        elif shap_values_2020.ndim == 3:
            sv = shap_values_2020[:, :, cls_idx]
        else:
            sv = shap_values_2020
        shap.summary_plot(sv, X_test_df_2020, show=False)
        plt.title(f'SHAP Feature Impact — {cls_name} (2020 - {best_algorithm})', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_beeswarm_{cls_name.lower().replace(" ", "_")}_2020.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9C. SHAP Dependence Plots — All Features (Built-Up Class index 0) ---
    print(f"\n--- SHAP Dependence Plots (2020) ---")
    if isinstance(shap_values_2020, list):
        mean_abs_shap_2020 = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values_2020], axis=0)
    elif shap_values_2020.ndim == 3:
        mean_abs_shap_2020 = np.abs(shap_values_2020).mean(axis=(0, 2))
    else:
        mean_abs_shap_2020 = np.abs(shap_values_2020).mean(axis=0)
        
    top_features_idx_2020 = np.argsort(mean_abs_shap_2020)[::-1]
    top_feature_names_2020 = [FEATURE_COLS_2020[idx] for idx in top_features_idx_2020]
    
    num_features_2020 = len(FEATURE_COLS_2020)
    num_cols = 3
    num_rows = (num_features_2020 + num_cols - 1) // num_cols
    
    cls_dep = 0
    if isinstance(shap_values_2020, list):
        sv_dep = shap_values_2020[cls_dep]
    elif shap_values_2020.ndim == 3:
        sv_dep = shap_values_2020[:, :, cls_dep]
    else:
        sv_dep = shap_values_2020
        
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 4.5 * num_rows))
    axes_flat = axes.flat
    
    for i, feat_idx in enumerate(top_features_idx_2020):
        ax = axes_flat[i]
        shap.dependence_plot(
            feat_idx, sv_dep, X_test_df_2020,
            interaction_index='auto', ax=ax, show=False
        )
        ax.set_title(f'{FEATURE_COLS_2020[feat_idx]}', fontsize=12, fontweight='bold')
        
    for i in range(num_features_2020, len(axes_flat)):
        fig.delaxes(axes_flat[i])
        
    plt.suptitle(f'SHAP Dependence Plots (All Features) — {CLASS_NAMES[cls_dep+1]} (2020)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('shap_dependence_all_features_2020.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9D. SHAP Waterfall Plots (One Per Class) ---
    print(f"\n--- SHAP Waterfall Plots (2020) ---")
    if best_algorithm == 'Random Forest':
        y_pred_test_2020 = y_pred_rf_2020
    elif best_algorithm == 'SVM':
        y_pred_test_2020 = y_pred_svm_2020
    else:
        y_pred_test_2020 = y_pred_xgb_2020
        
    for cls_idx in range(NUM_CLASSES_2020):
        cls_name = CLASS_NAMES[cls_idx + 1]
        cls_mask = y_test_2020 == cls_idx
        correct_mask = cls_mask & (y_pred_test_2020 == cls_idx)
        
        if correct_mask.sum() == 0:
            correct_mask = cls_mask
        if correct_mask.sum() == 0:
            continue
            
        sample_idx = np.where(correct_mask)[0][0]
        print(f"  Sample {sample_idx}: True={cls_name}, Predicted={CLASS_NAMES[y_pred_test_2020[sample_idx]+1]}")
        
        if isinstance(shap_values_2020, list):
            sv_sample = shap_values_2020[cls_idx][sample_idx]
            base_val = explainer_2020.expected_value[cls_idx]
        elif shap_values_2020.ndim == 3:
            sv_sample = shap_values_2020[sample_idx, :, cls_idx]
            base_val = explainer_2020.expected_value[cls_idx]
        else:
            sv_sample = shap_values_2020[sample_idx]
            base_val = explainer_2020.expected_value
            
        explanation = shap.Explanation(
            values=sv_sample,
            base_values=base_val,
            data=X_test_2020[sample_idx],
            feature_names=FEATURE_COLS_2020
        )
        
        fig = plt.figure(figsize=(10, 6))
        shap.plots.waterfall(explanation, show=False)
        plt.title(f'SHAP Waterfall — {cls_name} (Sample {sample_idx} - 2020)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_waterfall_{cls_name.lower().replace(" ", "_")}_2020.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9E. SHAP Force Plots ---
    # Built-up class force plot
    bu_idx = np.where((y_test_2020 == 0) & (y_pred_test_2020 == 0))[0]
    if len(bu_idx) > 0:
        idx = bu_idx[0]
        if isinstance(shap_values_2020, list):
            sv_force = shap_values_2020[0][idx]
            base_val = explainer_2020.expected_value[0]
        elif shap_values_2020.ndim == 3:
            sv_force = shap_values_2020[idx, :, 0]
            base_val = explainer_2020.expected_value[0]
        else:
            sv_force = shap_values_2020[idx]
            base_val = explainer_2020.expected_value
            
        print(f"\n--- SHAP Force Plot (Built-Up Sample - 2020) ---")
        shap.force_plot(base_val, sv_force, X_test_df_2020.iloc[idx], matplotlib=True, show=False)
        plt.title(f'SHAP Force Plot — Built-Up Prediction (2020)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('shap_force_plot_builtup_2020.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # All classes force plots
    for cls_idx in range(NUM_CLASSES_2020):
        cls_name = CLASS_NAMES[cls_idx + 1]
        correct_samples = np.where((y_test_2020 == cls_idx) & (y_pred_test_2020 == cls_idx))[0]
        
        if len(correct_samples) > 0:
            idx = correct_samples[0]
            if isinstance(shap_values_2020, list):
                sv_force = shap_values_2020[cls_idx][idx]
                base_val = explainer_2020.expected_value[cls_idx]
            elif shap_values_2020.ndim == 3:
                sv_force = shap_values_2020[idx, :, cls_idx]
                base_val = explainer_2020.expected_value[cls_idx]
            else:
                sv_force = shap_values_2020[idx]
                base_val = explainer_2020.expected_value
                
            print(f"\n--- SHAP Force Plot ({cls_name} Sample {idx} - 2020) ---")
            shap.force_plot(base_val, sv_force, X_test_df_2020.iloc[idx], matplotlib=True, show=False)
            plt.title(f'SHAP Force Plot — {cls_name} Prediction (2020)', fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'shap_force_plot_{cls_name.lower().replace(" ", "_")}_2020.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print(f"\nNo correctly classified sample found for class: {cls_name} (2020)")



## Step 8: Explainable AI (SHAP) - Year 2022

In [ ]:
# ============================================================================
# YEAR 2022 SHAP EXPLAINABILITY (Using Winning Algorithm: {best_algorithm})
# ============================================================================

if year_2022_available:
    print(f"\nComputing SHAP values for 2022 using {best_algorithm}...")
    
    # Retrieve winning model and test set for 2022
    if best_algorithm == 'Random Forest':
        model_2022 = rf_2022
    elif best_algorithm == 'SVM':
        model_2022 = svm_2022
    else:
        model_2022 = xgb_2022
        
    X_test_df_2022 = pd.DataFrame(X_test_2022, columns=FEATURE_COLS_2022)
    
    # TreeExplainer is fast for tree-based models; KernelExplainer is used for SVM
    if best_algorithm in ['XGBoost', 'Random Forest']:
        explainer_2022 = shap.TreeExplainer(model_2022)
        shap_values_2022 = explainer_2022.shap_values(X_test_2022)
    else:
        X_train_summary_2022 = shap.kmeans(X_train_2022, 20)
        explainer_2022 = shap.KernelExplainer(model_2022.predict_proba, X_train_summary_2022)
        shap_values_2022 = explainer_2022.shap_values(X_test_2022)
        
    print(f"✓ SHAP values computed for 2022 test samples.")
    if isinstance(shap_values_2022, list):
        print(f"  Number of class arrays: {len(shap_values_2022)}")
        print(f"  Shape per class: {shap_values_2022[0].shape}")
    elif isinstance(shap_values_2022, np.ndarray):
        print(f"  Shape: {shap_values_2022.shape}")
        
    # --- 9A. SHAP Bar Plot — Mean |SHAP| per feature across ALL classes ---
    print(f"\n--- SHAP Global Feature Importance (2022) ---")
    fig = plt.figure(figsize=(12, 8))
    target_names_2022 = [CLASS_NAMES[i+1] for i in range(NUM_CLASSES_2022)]
    shap.summary_plot(shap_values_2022, X_test_df_2022, plot_type='bar',
                      class_names=target_names_2022, show=False)
    plt.title(f'SHAP Feature Importance — All LULC Classes (2022 - {best_algorithm})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_summary_bar_all_classes_2022.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9B. SHAP Beeswarm Plot — Per Class ---
    for cls_idx in range(NUM_CLASSES_2022):
        cls_name = CLASS_NAMES[cls_idx + 1]
        print(f"\n--- SHAP Beeswarm: {cls_name} (2022) ---")
        fig = plt.figure(figsize=(12, 8))
        if isinstance(shap_values_2022, list):
            sv = shap_values_2022[cls_idx]
        elif shap_values_2022.ndim == 3:
            sv = shap_values_2022[:, :, cls_idx]
        else:
            sv = shap_values_2022
        shap.summary_plot(sv, X_test_df_2022, show=False)
        plt.title(f'SHAP Feature Impact — {cls_name} (2022 - {best_algorithm})', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_beeswarm_{cls_name.lower().replace(" ", "_")}_2022.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9C. SHAP Dependence Plots — All Features (Built-Up Class index 0) ---
    print(f"\n--- SHAP Dependence Plots (2022) ---")
    if isinstance(shap_values_2022, list):
        mean_abs_shap_2022 = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values_2022], axis=0)
    elif shap_values_2022.ndim == 3:
        mean_abs_shap_2022 = np.abs(shap_values_2022).mean(axis=(0, 2))
    else:
        mean_abs_shap_2022 = np.abs(shap_values_2022).mean(axis=0)
        
    top_features_idx_2022 = np.argsort(mean_abs_shap_2022)[::-1]
    top_feature_names_2022 = [FEATURE_COLS_2022[idx] for idx in top_features_idx_2022]
    
    num_features_2022 = len(FEATURE_COLS_2022)
    num_cols = 3
    num_rows = (num_features_2022 + num_cols - 1) // num_cols
    
    cls_dep = 0
    if isinstance(shap_values_2022, list):
        sv_dep = shap_values_2022[cls_dep]
    elif shap_values_2022.ndim == 3:
        sv_dep = shap_values_2022[:, :, cls_dep]
    else:
        sv_dep = shap_values_2022
        
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 4.5 * num_rows))
    axes_flat = axes.flat
    
    for i, feat_idx in enumerate(top_features_idx_2022):
        ax = axes_flat[i]
        shap.dependence_plot(
            feat_idx, sv_dep, X_test_df_2022,
            interaction_index='auto', ax=ax, show=False
        )
        ax.set_title(f'{FEATURE_COLS_2022[feat_idx]}', fontsize=12, fontweight='bold')
        
    for i in range(num_features_2022, len(axes_flat)):
        fig.delaxes(axes_flat[i])
        
    plt.suptitle(f'SHAP Dependence Plots (All Features) — {CLASS_NAMES[cls_dep+1]} (2022)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('shap_dependence_all_features_2022.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9D. SHAP Waterfall Plots (One Per Class) ---
    print(f"\n--- SHAP Waterfall Plots (2022) ---")
    if best_algorithm == 'Random Forest':
        y_pred_test_2022 = y_pred_rf_2022
    elif best_algorithm == 'SVM':
        y_pred_test_2022 = y_pred_svm_2022
    else:
        y_pred_test_2022 = y_pred_xgb_2022
        
    for cls_idx in range(NUM_CLASSES_2022):
        cls_name = CLASS_NAMES[cls_idx + 1]
        cls_mask = y_test_2022 == cls_idx
        correct_mask = cls_mask & (y_pred_test_2022 == cls_idx)
        
        if correct_mask.sum() == 0:
            correct_mask = cls_mask
        if correct_mask.sum() == 0:
            continue
            
        sample_idx = np.where(correct_mask)[0][0]
        print(f"  Sample {sample_idx}: True={cls_name}, Predicted={CLASS_NAMES[y_pred_test_2022[sample_idx]+1]}")
        
        if isinstance(shap_values_2022, list):
            sv_sample = shap_values_2022[cls_idx][sample_idx]
            base_val = explainer_2022.expected_value[cls_idx]
        elif shap_values_2022.ndim == 3:
            sv_sample = shap_values_2022[sample_idx, :, cls_idx]
            base_val = explainer_2022.expected_value[cls_idx]
        else:
            sv_sample = shap_values_2022[sample_idx]
            base_val = explainer_2022.expected_value
            
        explanation = shap.Explanation(
            values=sv_sample,
            base_values=base_val,
            data=X_test_2022[sample_idx],
            feature_names=FEATURE_COLS_2022
        )
        
        fig = plt.figure(figsize=(10, 6))
        shap.plots.waterfall(explanation, show=False)
        plt.title(f'SHAP Waterfall — {cls_name} (Sample {sample_idx} - 2022)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_waterfall_{cls_name.lower().replace(" ", "_")}_2022.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9E. SHAP Force Plots ---
    # Built-up class force plot
    bu_idx = np.where((y_test_2022 == 0) & (y_pred_test_2022 == 0))[0]
    if len(bu_idx) > 0:
        idx = bu_idx[0]
        if isinstance(shap_values_2022, list):
            sv_force = shap_values_2022[0][idx]
            base_val = explainer_2022.expected_value[0]
        elif shap_values_2022.ndim == 3:
            sv_force = shap_values_2022[idx, :, 0]
            base_val = explainer_2022.expected_value[0]
        else:
            sv_force = shap_values_2022[idx]
            base_val = explainer_2022.expected_value
            
        print(f"\n--- SHAP Force Plot (Built-Up Sample - 2022) ---")
        shap.force_plot(base_val, sv_force, X_test_df_2022.iloc[idx], matplotlib=True, show=False)
        plt.title(f'SHAP Force Plot — Built-Up Prediction (2022)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('shap_force_plot_builtup_2022.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # All classes force plots
    for cls_idx in range(NUM_CLASSES_2022):
        cls_name = CLASS_NAMES[cls_idx + 1]
        correct_samples = np.where((y_test_2022 == cls_idx) & (y_pred_test_2022 == cls_idx))[0]
        
        if len(correct_samples) > 0:
            idx = correct_samples[0]
            if isinstance(shap_values_2022, list):
                sv_force = shap_values_2022[cls_idx][idx]
                base_val = explainer_2022.expected_value[cls_idx]
            elif shap_values_2022.ndim == 3:
                sv_force = shap_values_2022[idx, :, cls_idx]
                base_val = explainer_2022.expected_value[cls_idx]
            else:
                sv_force = shap_values_2022[idx]
                base_val = explainer_2022.expected_value
                
            print(f"\n--- SHAP Force Plot ({cls_name} Sample {idx} - 2022) ---")
            shap.force_plot(base_val, sv_force, X_test_df_2022.iloc[idx], matplotlib=True, show=False)
            plt.title(f'SHAP Force Plot — {cls_name} Prediction (2022)', fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'shap_force_plot_{cls_name.lower().replace(" ", "_")}_2022.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print(f"\nNo correctly classified sample found for class: {cls_name} (2022)")



## Step 9: Explainable AI (SHAP) - Year 2024

In [ ]:
# ============================================================================
# YEAR 2024 SHAP EXPLAINABILITY (Using Winning Algorithm: {best_algorithm})
# ============================================================================

if year_2024_available:
    print(f"\nComputing SHAP values for 2024 using {best_algorithm}...")
    
    # Retrieve winning model and test set for 2024
    if best_algorithm == 'Random Forest':
        model_2024 = rf_2024
    elif best_algorithm == 'SVM':
        model_2024 = svm_2024
    else:
        model_2024 = xgb_2024
        
    X_test_df_2024 = pd.DataFrame(X_test_2024, columns=FEATURE_COLS_2024)
    
    # TreeExplainer is fast for tree-based models; KernelExplainer is used for SVM
    if best_algorithm in ['XGBoost', 'Random Forest']:
        explainer_2024 = shap.TreeExplainer(model_2024)
        shap_values_2024 = explainer_2024.shap_values(X_test_2024)
    else:
        X_train_summary_2024 = shap.kmeans(X_train_2024, 20)
        explainer_2024 = shap.KernelExplainer(model_2024.predict_proba, X_train_summary_2024)
        shap_values_2024 = explainer_2024.shap_values(X_test_2024)
        
    print(f"✓ SHAP values computed for 2024 test samples.")
    if isinstance(shap_values_2024, list):
        print(f"  Number of class arrays: {len(shap_values_2024)}")
        print(f"  Shape per class: {shap_values_2024[0].shape}")
    elif isinstance(shap_values_2024, np.ndarray):
        print(f"  Shape: {shap_values_2024.shape}")
        
    # --- 9A. SHAP Bar Plot — Mean |SHAP| per feature across ALL classes ---
    print(f"\n--- SHAP Global Feature Importance (2024) ---")
    fig = plt.figure(figsize=(12, 8))
    target_names_2024 = [CLASS_NAMES[i+1] for i in range(NUM_CLASSES_2024)]
    shap.summary_plot(shap_values_2024, X_test_df_2024, plot_type='bar',
                      class_names=target_names_2024, show=False)
    plt.title(f'SHAP Feature Importance — All LULC Classes (2024 - {best_algorithm})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_summary_bar_all_classes_2024.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9B. SHAP Beeswarm Plot — Per Class ---
    for cls_idx in range(NUM_CLASSES_2024):
        cls_name = CLASS_NAMES[cls_idx + 1]
        print(f"\n--- SHAP Beeswarm: {cls_name} (2024) ---")
        fig = plt.figure(figsize=(12, 8))
        if isinstance(shap_values_2024, list):
            sv = shap_values_2024[cls_idx]
        elif shap_values_2024.ndim == 3:
            sv = shap_values_2024[:, :, cls_idx]
        else:
            sv = shap_values_2024
        shap.summary_plot(sv, X_test_df_2024, show=False)
        plt.title(f'SHAP Feature Impact — {cls_name} (2024 - {best_algorithm})', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_beeswarm_{cls_name.lower().replace(" ", "_")}_2024.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9C. SHAP Dependence Plots — All Features (Built-Up Class index 0) ---
    print(f"\n--- SHAP Dependence Plots (2024) ---")
    if isinstance(shap_values_2024, list):
        mean_abs_shap_2024 = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values_2024], axis=0)
    elif shap_values_2024.ndim == 3:
        mean_abs_shap_2024 = np.abs(shap_values_2024).mean(axis=(0, 2))
    else:
        mean_abs_shap_2024 = np.abs(shap_values_2024).mean(axis=0)
        
    top_features_idx_2024 = np.argsort(mean_abs_shap_2024)[::-1]
    top_feature_names_2024 = [FEATURE_COLS_2024[idx] for idx in top_features_idx_2024]
    
    num_features_2024 = len(FEATURE_COLS_2024)
    num_cols = 3
    num_rows = (num_features_2024 + num_cols - 1) // num_cols
    
    cls_dep = 0
    if isinstance(shap_values_2024, list):
        sv_dep = shap_values_2024[cls_dep]
    elif shap_values_2024.ndim == 3:
        sv_dep = shap_values_2024[:, :, cls_dep]
    else:
        sv_dep = shap_values_2024
        
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 4.5 * num_rows))
    axes_flat = axes.flat
    
    for i, feat_idx in enumerate(top_features_idx_2024):
        ax = axes_flat[i]
        shap.dependence_plot(
            feat_idx, sv_dep, X_test_df_2024,
            interaction_index='auto', ax=ax, show=False
        )
        ax.set_title(f'{FEATURE_COLS_2024[feat_idx]}', fontsize=12, fontweight='bold')
        
    for i in range(num_features_2024, len(axes_flat)):
        fig.delaxes(axes_flat[i])
        
    plt.suptitle(f'SHAP Dependence Plots (All Features) — {CLASS_NAMES[cls_dep+1]} (2024)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('shap_dependence_all_features_2024.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- 9D. SHAP Waterfall Plots (One Per Class) ---
    print(f"\n--- SHAP Waterfall Plots (2024) ---")
    if best_algorithm == 'Random Forest':
        y_pred_test_2024 = y_pred_rf_2024
    elif best_algorithm == 'SVM':
        y_pred_test_2024 = y_pred_svm_2024
    else:
        y_pred_test_2024 = y_pred_xgb_2024
        
    for cls_idx in range(NUM_CLASSES_2024):
        cls_name = CLASS_NAMES[cls_idx + 1]
        cls_mask = y_test_2024 == cls_idx
        correct_mask = cls_mask & (y_pred_test_2024 == cls_idx)
        
        if correct_mask.sum() == 0:
            correct_mask = cls_mask
        if correct_mask.sum() == 0:
            continue
            
        sample_idx = np.where(correct_mask)[0][0]
        print(f"  Sample {sample_idx}: True={cls_name}, Predicted={CLASS_NAMES[y_pred_test_2024[sample_idx]+1]}")
        
        if isinstance(shap_values_2024, list):
            sv_sample = shap_values_2024[cls_idx][sample_idx]
            base_val = explainer_2024.expected_value[cls_idx]
        elif shap_values_2024.ndim == 3:
            sv_sample = shap_values_2024[sample_idx, :, cls_idx]
            base_val = explainer_2024.expected_value[cls_idx]
        else:
            sv_sample = shap_values_2024[sample_idx]
            base_val = explainer_2024.expected_value
            
        explanation = shap.Explanation(
            values=sv_sample,
            base_values=base_val,
            data=X_test_2024[sample_idx],
            feature_names=FEATURE_COLS_2024
        )
        
        fig = plt.figure(figsize=(10, 6))
        shap.plots.waterfall(explanation, show=False)
        plt.title(f'SHAP Waterfall — {cls_name} (Sample {sample_idx} - 2024)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_waterfall_{cls_name.lower().replace(" ", "_")}_2024.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # --- 9E. SHAP Force Plots ---
    # Built-up class force plot
    bu_idx = np.where((y_test_2024 == 0) & (y_pred_test_2024 == 0))[0]
    if len(bu_idx) > 0:
        idx = bu_idx[0]
        if isinstance(shap_values_2024, list):
            sv_force = shap_values_2024[0][idx]
            base_val = explainer_2024.expected_value[0]
        elif shap_values_2024.ndim == 3:
            sv_force = shap_values_2024[idx, :, 0]
            base_val = explainer_2024.expected_value[0]
        else:
            sv_force = shap_values_2024[idx]
            base_val = explainer_2024.expected_value
            
        print(f"\n--- SHAP Force Plot (Built-Up Sample - 2024) ---")
        shap.force_plot(base_val, sv_force, X_test_df_2024.iloc[idx], matplotlib=True, show=False)
        plt.title(f'SHAP Force Plot — Built-Up Prediction (2024)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('shap_force_plot_builtup_2024.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    # All classes force plots
    for cls_idx in range(NUM_CLASSES_2024):
        cls_name = CLASS_NAMES[cls_idx + 1]
        correct_samples = np.where((y_test_2024 == cls_idx) & (y_pred_test_2024 == cls_idx))[0]
        
        if len(correct_samples) > 0:
            idx = correct_samples[0]
            if isinstance(shap_values_2024, list):
                sv_force = shap_values_2024[cls_idx][idx]
                base_val = explainer_2024.expected_value[cls_idx]
            elif shap_values_2024.ndim == 3:
                sv_force = shap_values_2024[idx, :, cls_idx]
                base_val = explainer_2024.expected_value[cls_idx]
            else:
                sv_force = shap_values_2024[idx]
                base_val = explainer_2024.expected_value
                
            print(f"\n--- SHAP Force Plot ({cls_name} Sample {idx} - 2024) ---")
            shap.force_plot(base_val, sv_force, X_test_df_2024.iloc[idx], matplotlib=True, show=False)
            plt.title(f'SHAP Force Plot — {cls_name} Prediction (2024)', fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'shap_force_plot_{cls_name.lower().replace(" ", "_")}_2024.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print(f"\nNo correctly classified sample found for class: {cls_name} (2024)")



## Step 10: Multi-Temporal Explanation Consistency Analysis

In [ ]:
print("\nEvaluating feature explanation consistency across time...")
shap_history = {}

for year in [2020, 2022, 2024]:
    # Check if shap_values for that year exists globally
    glob_name = f'shap_values_{year}'
    feat_name = f'FEATURE_COLS_{year}'
    if glob_name in globals() and feat_name in globals():
        sv = globals()[glob_name]
        feats = globals()[feat_name]
        
        if isinstance(sv, list):
            mean_abs = np.mean([np.abs(a).mean(axis=0) for a in sv], axis=0)
        elif sv.ndim == 3:
            mean_abs = np.abs(sv).mean(axis=(0, 2))
        else:
            mean_abs = np.abs(sv).mean(axis=0)
            
        shap_history[year] = pd.Series(mean_abs, index=feats)

if shap_history:
    consistency_df = pd.DataFrame(shap_history).fillna(0.0)
    consistency_df['Mean_Across_Years'] = consistency_df.mean(axis=1)
    consistency_df = consistency_df.sort_values('Mean_Across_Years', ascending=False)
    consistency_df = consistency_df.drop(columns=['Mean_Across_Years'])
    
    consistency_df.to_csv(os.path.join(DATA_DIR, 'shap_temporal_consistency.csv'))
    
    # Plot Grouped Bar Chart
    fig, ax = plt.subplots(figsize=(14, 8))
    melted_df = consistency_df.reset_index().rename(columns={'index': 'Feature'}).melt(
        id_vars='Feature', var_name='Year', value_name='Mean |SHAP|'
    )
    
    sns.barplot(
        data=melted_df,
        x='Mean |SHAP|',
        y='Feature',
        hue='Year',
        ax=ax,
        edgecolor='black',
        linewidth=0.5
    )
    ax.set_title('SHAP Feature Importance Consistency Across Years (2020, 2022, 2024)', fontsize=16, fontweight='bold')
    ax.set_xlabel('Mean Absolute SHAP Value (Predictive Contribution)')
    ax.set_ylabel('Features')
    plt.tight_layout()
    plt.savefig('shap_temporal_consistency_barplot.png', dpi=300, bbox_inches='tight')
    plt.show()

## Step 11: Reconstruct LULC Map from TFRecords - Year 2020

In [ ]:
# ============================================================================
# YEAR 2020 SPATIAL CLASSIFICATION MAPPING (PREDICTION ON TFRECORD PATTERNS)
# ============================================================================

if RASTERIO_AVAILABLE and year_2020_available:
    try:
        import tensorflow as tf
        TENSORFLOW_AVAILABLE = True
    except ImportError:
        TENSORFLOW_AVAILABLE = False
        print("Warning: 'tensorflow' not found. TFRecord parsing will fail.")
        
    if TENSORFLOW_AVAILABLE:
        print(f"\nReconstructing LULC Map for 2020 using {best_algorithm}...")
        
        # Retrieve winning model and test features
        if best_algorithm == 'Random Forest':
            model_2020 = rf_2020
        elif best_algorithm == 'SVM':
            model_2020 = svm_2020
        else:
            model_2020 = xgb_2020
            
        tfrecord_files = sorted(glob.glob(os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2020_TFRecord-*.tfrecord.gz')))
        json_file = os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2020_TFRecord-mixer.json')
        output_tiff = os.path.join(DATA_DIR, f'GAMA_LULC_{best_algorithm.replace(" ", "_")}_2020.tif')
        
        if not tfrecord_files or not os.path.exists(json_file):
            print(f"⚠ TFRecord shards or JSON mixer metadata not found for 2020!")
        else:
            with open(json_file, 'r') as f:
                mixer = json.load(f)
                
            projection = mixer['projection']
            affine = projection['affine']['doubleMatrix']
            crs = projection['crs']
            
            patch_cols = mixer['patchDimensions'][0]
            patch_rows = mixer['patchDimensions'][1]
            patches_along_cols = mixer['patchesPerRow']
            total_patches = mixer['totalPatches']
            
            total_cols = patches_along_cols * patch_cols
            total_rows = (total_patches // patches_along_cols) * patch_rows
            
            if 'features' in mixer:
                bands = [b['name'] for b in mixer['features']]
            else:
                bands = [col for col in df_2020.columns if col not in [LABEL_COL_2020, 'label_0idx']]
                
            feature_description = {
                b: tf.io.FixedLenFeature(shape=[patch_rows * patch_cols], dtype=tf.float32)
                for b in bands
            }
            
            def parse_tfrecord(example_proto):
                return tf.io.parse_single_example(example_proto, feature_description)
                
            dataset = tf.data.TFRecordDataset(tfrecord_files, compression_type='GZIP')
            parsed_dataset = dataset.map(parse_tfrecord)
            
            prediction_grid = np.zeros((total_rows, total_cols), dtype=np.uint8)
            
            import gc
            for patch_idx, parsed_patch in enumerate(parsed_dataset):
                col_offset = (patch_idx % patches_along_cols) * patch_cols
                row_offset = (patch_idx // patches_along_cols) * patch_rows
                
                actual_rows = min(patch_rows, total_rows - row_offset)
                actual_cols = min(patch_cols, total_cols - col_offset)
                
                features_dict = {b: parsed_patch[b].numpy() for b in FEATURE_COLS_2020}
                pixels = np.stack([features_dict[feat] for feat in FEATURE_COLS_2020], axis=-1)
                
                # Check band for validation mask (B2 or B3)
                b_mask_name = 'B2' if 'B2' in parsed_patch else 'B3'
                b_mask = parsed_patch[b_mask_name].numpy()
                valid_mask = (b_mask > 0) & (~np.isnan(b_mask))
                
                predictions = np.zeros(patch_rows * patch_cols, dtype=np.uint8)
                valid_pixels = pixels[valid_mask]
                
                if len(valid_pixels) > 0:
                    preds = model_2020.predict(valid_pixels) + 1  # 0-indexed -> 1-indexed
                    predictions[valid_mask] = preds.astype(np.uint8)
                    
                patch_preds_2d = predictions.reshape(patch_rows, patch_cols)
                prediction_grid[
                    row_offset:row_offset+actual_rows,
                    col_offset:col_offset+actual_cols
                ] = patch_preds_2d[:actual_rows, :actual_cols]
                
                if (patch_idx + 1) % 150 == 0:
                    gc.collect()
                    
            transform = rasterio.transform.Affine(
                affine[0], affine[1], affine[2],
                affine[3], affine[4], affine[5]
            )
            
            out_meta = {
                'driver': 'GTiff',
                'height': total_rows,
                'width': total_cols,
                'count': 1,
                'dtype': 'uint8',
                'crs': crs,
                'transform': transform,
                'nodata': 0,
                'compress': 'deflate'
            }
            
            with rasterio.open(output_tiff, 'w', **out_meta) as dst:
                dst.write(prediction_grid, 1)
                
            print(f"✓ Seamless GAMA LULC map saved: {os.path.basename(output_tiff)}")
            
            # --- Spatial stats printout ---
            pixel_area_km2 = (10 * 10) / 1e6
            print(f"\n--- LULC Class Areas (2020 - {best_algorithm}) ---")
            for cls_id in [1, 2, 3, 4]:
                cls_name = CLASS_NAMES[cls_id]
                pixel_count = np.sum(prediction_grid == cls_id)
                area_km2 = pixel_count * pixel_area_km2
                pct = pixel_count / np.sum(prediction_grid > 0) * 100 if np.sum(prediction_grid > 0) > 0 else 0
                print(f"  {cls_name:12s}: {pixel_count:>10,} pixels = {area_km2:>8.2f} km²  ({pct:.1f}%)")
                
            # Plot the final classified LULC map
            from matplotlib.colors import ListedColormap
            from matplotlib.patches import Patch
            
            map_colors = ['#FFFFFF', '#E8453C', '#1B7A2F', '#2196F3', '#9E9E9E']
            cmap = ListedColormap(map_colors)
            
            fig, ax = plt.subplots(figsize=(14, 12))
            im = ax.imshow(prediction_grid, cmap=cmap, vmin=0, vmax=4)
            ax.set_title(f'{best_algorithm} Classified LULC Map — GAMA, Ghana (2020)', fontsize=16, fontweight='bold')
            ax.axis('off')
            
            legend_elements = [
                Patch(facecolor='#E8453C', edgecolor='black', label='Built-Up'),
                Patch(facecolor='#1B7A2F', edgecolor='black', label='Vegetation'),
                Patch(facecolor='#2196F3', edgecolor='black', label='Water'),
                Patch(facecolor='#9E9E9E', edgecolor='black', label='Bare Soil'),
            ]
            ax.legend(handles=legend_elements, loc='lower right', fontsize=12, framealpha=0.9, edgecolor='black', title='LULC Classes', title_fontsize=13)
            plt.tight_layout()
            plt.savefig(f'classified_map_2020.png', dpi=300, bbox_inches='tight')
            plt.show()



## Step 12: Reconstruct LULC Map from TFRecords - Year 2022

In [ ]:
# ============================================================================
# YEAR 2022 SPATIAL CLASSIFICATION MAPPING (PREDICTION ON TFRECORD PATTERNS)
# ============================================================================

if RASTERIO_AVAILABLE and year_2022_available:
    try:
        import tensorflow as tf
        TENSORFLOW_AVAILABLE = True
    except ImportError:
        TENSORFLOW_AVAILABLE = False
        print("Warning: 'tensorflow' not found. TFRecord parsing will fail.")
        
    if TENSORFLOW_AVAILABLE:
        print(f"\nReconstructing LULC Map for 2022 using {best_algorithm}...")
        
        # Retrieve winning model and test features
        if best_algorithm == 'Random Forest':
            model_2022 = rf_2022
        elif best_algorithm == 'SVM':
            model_2022 = svm_2022
        else:
            model_2022 = xgb_2022
            
        tfrecord_files = sorted(glob.glob(os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2022_TFRecord-*.tfrecord.gz')))
        json_file = os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2022_TFRecord-mixer.json')
        output_tiff = os.path.join(DATA_DIR, f'GAMA_LULC_{best_algorithm.replace(" ", "_")}_2022.tif')
        
        if not tfrecord_files or not os.path.exists(json_file):
            print(f"⚠ TFRecord shards or JSON mixer metadata not found for 2022!")
        else:
            with open(json_file, 'r') as f:
                mixer = json.load(f)
                
            projection = mixer['projection']
            affine = projection['affine']['doubleMatrix']
            crs = projection['crs']
            
            patch_cols = mixer['patchDimensions'][0]
            patch_rows = mixer['patchDimensions'][1]
            patches_along_cols = mixer['patchesPerRow']
            total_patches = mixer['totalPatches']
            
            total_cols = patches_along_cols * patch_cols
            total_rows = (total_patches // patches_along_cols) * patch_rows
            
            if 'features' in mixer:
                bands = [b['name'] for b in mixer['features']]
            else:
                bands = [col for col in df_2022.columns if col not in [LABEL_COL_2022, 'label_0idx']]
                
            feature_description = {
                b: tf.io.FixedLenFeature(shape=[patch_rows * patch_cols], dtype=tf.float32)
                for b in bands
            }
            
            def parse_tfrecord(example_proto):
                return tf.io.parse_single_example(example_proto, feature_description)
                
            dataset = tf.data.TFRecordDataset(tfrecord_files, compression_type='GZIP')
            parsed_dataset = dataset.map(parse_tfrecord)
            
            prediction_grid = np.zeros((total_rows, total_cols), dtype=np.uint8)
            
            import gc
            for patch_idx, parsed_patch in enumerate(parsed_dataset):
                col_offset = (patch_idx % patches_along_cols) * patch_cols
                row_offset = (patch_idx // patches_along_cols) * patch_rows
                
                actual_rows = min(patch_rows, total_rows - row_offset)
                actual_cols = min(patch_cols, total_cols - col_offset)
                
                features_dict = {b: parsed_patch[b].numpy() for b in FEATURE_COLS_2022}
                pixels = np.stack([features_dict[feat] for feat in FEATURE_COLS_2022], axis=-1)
                
                # Check band for validation mask (B2 or B3)
                b_mask_name = 'B2' if 'B2' in parsed_patch else 'B3'
                b_mask = parsed_patch[b_mask_name].numpy()
                valid_mask = (b_mask > 0) & (~np.isnan(b_mask))
                
                predictions = np.zeros(patch_rows * patch_cols, dtype=np.uint8)
                valid_pixels = pixels[valid_mask]
                
                if len(valid_pixels) > 0:
                    preds = model_2022.predict(valid_pixels) + 1  # 0-indexed -> 1-indexed
                    predictions[valid_mask] = preds.astype(np.uint8)
                    
                patch_preds_2d = predictions.reshape(patch_rows, patch_cols)
                prediction_grid[
                    row_offset:row_offset+actual_rows,
                    col_offset:col_offset+actual_cols
                ] = patch_preds_2d[:actual_rows, :actual_cols]
                
                if (patch_idx + 1) % 150 == 0:
                    gc.collect()
                    
            transform = rasterio.transform.Affine(
                affine[0], affine[1], affine[2],
                affine[3], affine[4], affine[5]
            )
            
            out_meta = {
                'driver': 'GTiff',
                'height': total_rows,
                'width': total_cols,
                'count': 1,
                'dtype': 'uint8',
                'crs': crs,
                'transform': transform,
                'nodata': 0,
                'compress': 'deflate'
            }
            
            with rasterio.open(output_tiff, 'w', **out_meta) as dst:
                dst.write(prediction_grid, 1)
                
            print(f"✓ Seamless GAMA LULC map saved: {os.path.basename(output_tiff)}")
            
            # --- Spatial stats printout ---
            pixel_area_km2 = (10 * 10) / 1e6
            print(f"\n--- LULC Class Areas (2022 - {best_algorithm}) ---")
            for cls_id in [1, 2, 3, 4]:
                cls_name = CLASS_NAMES[cls_id]
                pixel_count = np.sum(prediction_grid == cls_id)
                area_km2 = pixel_count * pixel_area_km2
                pct = pixel_count / np.sum(prediction_grid > 0) * 100 if np.sum(prediction_grid > 0) > 0 else 0
                print(f"  {cls_name:12s}: {pixel_count:>10,} pixels = {area_km2:>8.2f} km²  ({pct:.1f}%)")
                
            # Plot the final classified LULC map
            from matplotlib.colors import ListedColormap
            from matplotlib.patches import Patch
            
            map_colors = ['#FFFFFF', '#E8453C', '#1B7A2F', '#2196F3', '#9E9E9E']
            cmap = ListedColormap(map_colors)
            
            fig, ax = plt.subplots(figsize=(14, 12))
            im = ax.imshow(prediction_grid, cmap=cmap, vmin=0, vmax=4)
            ax.set_title(f'{best_algorithm} Classified LULC Map — GAMA, Ghana (2022)', fontsize=16, fontweight='bold')
            ax.axis('off')
            
            legend_elements = [
                Patch(facecolor='#E8453C', edgecolor='black', label='Built-Up'),
                Patch(facecolor='#1B7A2F', edgecolor='black', label='Vegetation'),
                Patch(facecolor='#2196F3', edgecolor='black', label='Water'),
                Patch(facecolor='#9E9E9E', edgecolor='black', label='Bare Soil'),
            ]
            ax.legend(handles=legend_elements, loc='lower right', fontsize=12, framealpha=0.9, edgecolor='black', title='LULC Classes', title_fontsize=13)
            plt.tight_layout()
            plt.savefig(f'classified_map_2022.png', dpi=300, bbox_inches='tight')
            plt.show()



## Step 13: Reconstruct LULC Map from TFRecords - Year 2024

In [ ]:
# ============================================================================
# YEAR 2024 SPATIAL CLASSIFICATION MAPPING (PREDICTION ON TFRECORD PATTERNS)
# ============================================================================

if RASTERIO_AVAILABLE and year_2024_available:
    try:
        import tensorflow as tf
        TENSORFLOW_AVAILABLE = True
    except ImportError:
        TENSORFLOW_AVAILABLE = False
        print("Warning: 'tensorflow' not found. TFRecord parsing will fail.")
        
    if TENSORFLOW_AVAILABLE:
        print(f"\nReconstructing LULC Map for 2024 using {best_algorithm}...")
        
        # Retrieve winning model and test features
        if best_algorithm == 'Random Forest':
            model_2024 = rf_2024
        elif best_algorithm == 'SVM':
            model_2024 = svm_2024
        else:
            model_2024 = xgb_2024
            
        tfrecord_files = sorted(glob.glob(os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2024_TFRecord-*.tfrecord.gz')))
        json_file = os.path.join(DATA_DIR, 'XAI_LULC_GAMA_FeatureStack_2024_TFRecord-mixer.json')
        output_tiff = os.path.join(DATA_DIR, f'GAMA_LULC_{best_algorithm.replace(" ", "_")}_2024.tif')
        
        if not tfrecord_files or not os.path.exists(json_file):
            print(f"⚠ TFRecord shards or JSON mixer metadata not found for 2024!")
        else:
            with open(json_file, 'r') as f:
                mixer = json.load(f)
                
            projection = mixer['projection']
            affine = projection['affine']['doubleMatrix']
            crs = projection['crs']
            
            patch_cols = mixer['patchDimensions'][0]
            patch_rows = mixer['patchDimensions'][1]
            patches_along_cols = mixer['patchesPerRow']
            total_patches = mixer['totalPatches']
            
            total_cols = patches_along_cols * patch_cols
            total_rows = (total_patches // patches_along_cols) * patch_rows
            
            if 'features' in mixer:
                bands = [b['name'] for b in mixer['features']]
            else:
                bands = [col for col in df_2024.columns if col not in [LABEL_COL_2024, 'label_0idx']]
                
            feature_description = {
                b: tf.io.FixedLenFeature(shape=[patch_rows * patch_cols], dtype=tf.float32)
                for b in bands
            }
            
            def parse_tfrecord(example_proto):
                return tf.io.parse_single_example(example_proto, feature_description)
                
            dataset = tf.data.TFRecordDataset(tfrecord_files, compression_type='GZIP')
            parsed_dataset = dataset.map(parse_tfrecord)
            
            prediction_grid = np.zeros((total_rows, total_cols), dtype=np.uint8)
            
            import gc
            for patch_idx, parsed_patch in enumerate(parsed_dataset):
                col_offset = (patch_idx % patches_along_cols) * patch_cols
                row_offset = (patch_idx // patches_along_cols) * patch_rows
                
                actual_rows = min(patch_rows, total_rows - row_offset)
                actual_cols = min(patch_cols, total_cols - col_offset)
                
                features_dict = {b: parsed_patch[b].numpy() for b in FEATURE_COLS_2024}
                pixels = np.stack([features_dict[feat] for feat in FEATURE_COLS_2024], axis=-1)
                
                # Check band for validation mask (B2 or B3)
                b_mask_name = 'B2' if 'B2' in parsed_patch else 'B3'
                b_mask = parsed_patch[b_mask_name].numpy()
                valid_mask = (b_mask > 0) & (~np.isnan(b_mask))
                
                predictions = np.zeros(patch_rows * patch_cols, dtype=np.uint8)
                valid_pixels = pixels[valid_mask]
                
                if len(valid_pixels) > 0:
                    preds = model_2024.predict(valid_pixels) + 1  # 0-indexed -> 1-indexed
                    predictions[valid_mask] = preds.astype(np.uint8)
                    
                patch_preds_2d = predictions.reshape(patch_rows, patch_cols)
                prediction_grid[
                    row_offset:row_offset+actual_rows,
                    col_offset:col_offset+actual_cols
                ] = patch_preds_2d[:actual_rows, :actual_cols]
                
                if (patch_idx + 1) % 150 == 0:
                    gc.collect()
                    
            transform = rasterio.transform.Affine(
                affine[0], affine[1], affine[2],
                affine[3], affine[4], affine[5]
            )
            
            out_meta = {
                'driver': 'GTiff',
                'height': total_rows,
                'width': total_cols,
                'count': 1,
                'dtype': 'uint8',
                'crs': crs,
                'transform': transform,
                'nodata': 0,
                'compress': 'deflate'
            }
            
            with rasterio.open(output_tiff, 'w', **out_meta) as dst:
                dst.write(prediction_grid, 1)
                
            print(f"✓ Seamless GAMA LULC map saved: {os.path.basename(output_tiff)}")
            
            # --- Spatial stats printout ---
            pixel_area_km2 = (10 * 10) / 1e6
            print(f"\n--- LULC Class Areas (2024 - {best_algorithm}) ---")
            for cls_id in [1, 2, 3, 4]:
                cls_name = CLASS_NAMES[cls_id]
                pixel_count = np.sum(prediction_grid == cls_id)
                area_km2 = pixel_count * pixel_area_km2
                pct = pixel_count / np.sum(prediction_grid > 0) * 100 if np.sum(prediction_grid > 0) > 0 else 0
                print(f"  {cls_name:12s}: {pixel_count:>10,} pixels = {area_km2:>8.2f} km²  ({pct:.1f}%)")
                
            # Plot the final classified LULC map
            from matplotlib.colors import ListedColormap
            from matplotlib.patches import Patch
            
            map_colors = ['#FFFFFF', '#E8453C', '#1B7A2F', '#2196F3', '#9E9E9E']
            cmap = ListedColormap(map_colors)
            
            fig, ax = plt.subplots(figsize=(14, 12))
            im = ax.imshow(prediction_grid, cmap=cmap, vmin=0, vmax=4)
            ax.set_title(f'{best_algorithm} Classified LULC Map — GAMA, Ghana (2024)', fontsize=16, fontweight='bold')
            ax.axis('off')
            
            legend_elements = [
                Patch(facecolor='#E8453C', edgecolor='black', label='Built-Up'),
                Patch(facecolor='#1B7A2F', edgecolor='black', label='Vegetation'),
                Patch(facecolor='#2196F3', edgecolor='black', label='Water'),
                Patch(facecolor='#9E9E9E', edgecolor='black', label='Bare Soil'),
            ]
            ax.legend(handles=legend_elements, loc='lower right', fontsize=12, framealpha=0.9, edgecolor='black', title='LULC Classes', title_fontsize=13)
            plt.tight_layout()
            plt.savefig(f'classified_map_2024.png', dpi=300, bbox_inches='tight')
            plt.show()

